# NB27 · Concurrency & Asyncio

`advanced/` · 20 questions · the last one is a story project

Doing several things at once: threads for blocking work, `asyncio` for
lots of waiting. Jupyter already runs an event loop, so use top-level
`await` in a cell instead of `asyncio.run()`. Every wait in this notebook
is a few milliseconds, and every result is collected and printed in a fixed
order, so your output is always the same.

## You'll need

- NB22: generators
- NB24: context managers

## You'll practise

- Concurrency vs parallelism; the GIL and the free-threaded build (3.13+)
- threading.Thread, join, race conditions, Lock; queue.Queue
- concurrent.futures: ThreadPoolExecutor map vs submit/as_completed
- ProcessPoolExecutor in concept (the notebook pickling caveat)
- asyncio: coroutines, await, top-level await in Jupyter, gather, create_task
- TaskGroup (3.11+), asyncio.timeout, Semaphore, to_thread
- Async iterators, async generators, async with

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Questions get longer as you go: one-liners first, a small story project last. Every solution only uses what this notebook and the ones before it teach.

---

## Warm-up · one line each

### Q01 · Real-life

`send_email` is slow in real life. Run `send_email("ana@example.com")` in
its own thread, wait for the thread to finish, then print `log`.

**Expected output:**

```text
['sent to ana@example.com']
```

<details><summary><b>Solution</b></summary>

```python
import threading
t = threading.Thread(target=send_email, args=("ana@example.com",))
t.start()
t.join()
print(log)
```

</details>

In [ ]:
log = []

def send_email(address):
    log.append("sent to " + address)

# your code here


### Q02 · Puzzle

Start the three fetches below at the same time, in the order slow, fast,
medium, and print the list of results you get back from one `await`.

**Expected output:**

```text
['slow', 'fast', 'medium']
```

<details><summary><b>Solution</b></summary>

```python
results = await asyncio.gather(fetch("slow", 0.04), fetch("fast", 0.01), fetch("medium", 0.02))
print(results)
```

</details>

In [ ]:
import asyncio

async def fetch(name, delay):
    await asyncio.sleep(delay)
    return name

# your code here


### Q03 · Real-life

A shop applies the same discount to every price. Use a thread pool with 3
workers to run `apply_discount` over `prices`, and print the results as a list.

**Expected output:**

```text
[18.0, 31.95, 45.0]
```

<details><summary><b>Solution</b></summary>

```python
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=3) as pool:
    print(list(pool.map(apply_discount, prices)))
```

</details>

In [ ]:
prices = [20, 35.5, 50]

def apply_discount(price):
    return round(price * 0.9, 2)

# your code here


### Q04 · Output-match

Turn `slow_square(7)` into a task so it starts running in the background.
Print `started` straight away, then print the task's result.

**Expected output:**

```text
started
49
```

<details><summary><b>Solution</b></summary>

```python
task = asyncio.create_task(slow_square(7))
print("started")
print(await task)
```

</details>

In [ ]:
import asyncio

async def slow_square(n):
    await asyncio.sleep(0.01)
    return n * n

# your code here


### Q05 · Real-life

`read_report` blocks for a moment (`time.sleep`), which would freeze the
event loop. Run it in a worker thread from async code and print what it returns.

**Expected output:**

```text
report ready
```

<details><summary><b>Solution</b></summary>

```python
print(await asyncio.to_thread(read_report))
```

</details>

In [ ]:
import asyncio
import time

def read_report():
    time.sleep(0.01)
    return "report ready"

# your code here


---

## Combine · 2 to 4 lines

### Q06 · Real-life

A weather API can hang. Give `slow_api()` at most 0.01 seconds: print its
data if it answers in time, otherwise print `timed out`.

**Expected output:**

```text
timed out
```

<details><summary><b>Solution</b></summary>

```python
try:
    async with asyncio.timeout(0.01):
        print(await slow_api())
except TimeoutError:
    print("timed out")
```

</details>

In [ ]:
import asyncio

async def slow_api():
    await asyncio.sleep(0.05)
    return "data"

# your code here


### Q07 · Real-life

Check all `items` with a thread pool using `submit`, then print one line
per item (the item and whether it is in stock) in the order of `items`.

**Expected output:**

```text
plum True
apple True
pear False
```

<details><summary><b>Solution</b></summary>

```python
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor() as pool:
    futures = [pool.submit(in_stock, item) for item in items]
    for future in futures:
        print(*future.result())
```

</details>

In [ ]:
stock = {"apple": 5, "pear": 0, "plum": 12}
items = ["plum", "apple", "pear"]

def in_stock(item):
    return item, stock[item] > 0

# your code here


### Q08 · Build

Check every url concurrently. One failing check must not cancel the others.
Print one line per url, in order: the url and either `ok` or
`failed:` plus the error message.

**Expected output:**

```text
a.com ok
bad.org failed: unreachable bad.org
c.net ok
```

<details><summary><b>Solution</b></summary>

```python
results = await asyncio.gather(*(check(url) for url in urls), return_exceptions=True)
for url, result in zip(urls, results):
    if isinstance(result, Exception):
        print(url, "failed:", result)
    else:
        print(url, result)
```

</details>

In [ ]:
import asyncio

async def check(url):
    await asyncio.sleep(0.01)
    if "bad" in url:
        raise ValueError("unreachable " + url)
    return "ok"

urls = ["a.com", "bad.org", "c.net"]

# your code here


### Q09 · Build

Use a `TaskGroup` to price every item concurrently (one task per item).
When the group has finished, print the total price.

**Expected output:**

```text
130
```

<details><summary><b>Solution</b></summary>

```python
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(price_of(item)) for item in items]
print(sum(task.result() for task in tasks))
```

</details>

In [ ]:
import asyncio

async def price_of(item):
    await asyncio.sleep(0.01)
    return len(item) * 10

items = ["tea", "coffee", "cake"]

# your code here


### Q10 · Output-match

Write an async generator `countdown(n)` that sleeps 0.001 seconds and then
yields `n`, `n - 1`, ... down to 1. Loop over `countdown(3)` with
`async for`, printing each number on its own line, then print `Liftoff!`.

**Expected output:**

```text
3
2
1
Liftoff!
```

<details><summary><b>Solution</b></summary>

```python
async def countdown(n):
    while n > 0:
        await asyncio.sleep(0.001)
        yield n
        n -= 1

async for number in countdown(3):
    print(number)
print("Liftoff!")
```

</details>

In [ ]:
import asyncio

# your code here


---

## Small real tasks · 5 to 8 lines

### Q11 · Real-life

A laundry worker thread takes jobs from a `queue.Queue`. The main thread
puts every job in `jobs` on the queue, then a `None` to say "no more".
The worker appends each job in upper case to `done` and stops at `None`.
Wait for the worker, then print `done`.

**Expected output:**

```text
['WASH', 'DRY', 'FOLD', 'IRON']
```

<details><summary><b>Solution</b></summary>

```python
tasks = queue.Queue()

def worker():
    while True:
        job = tasks.get()
        if job is None:
            break
        done.append(job.upper())

t = threading.Thread(target=worker)
t.start()
for job in jobs:
    tasks.put(job)
tasks.put(None)
t.join()
print(done)
```

</details>

In [ ]:
import queue
import threading

jobs = ["wash", "dry", "fold", "iron"]
done = []

# your code here


### Q12 · Real-life

A download server may only serve 2 downloads at once. Download every file
in `files` concurrently without ever exceeding that limit. Print the list
of results (in the order of `files`), then the highest number of downloads
that were active at the same time, labelled like the expected output.

**Expected output:**

```text
['A', 'B', 'C', 'D', 'E']
Peak: 2
```

<details><summary><b>Solution</b></summary>

```python
limit = asyncio.Semaphore(2)

async def limited(name):
    async with limit:
        return await download(name)

results = await asyncio.gather(*(limited(name) for name in files))
print(results)
print("Peak:", peak)
```

</details>

In [ ]:
import asyncio

active = 0
peak = 0

async def download(name):
    global active, peak
    active += 1
    peak = max(peak, active)
    await asyncio.sleep(0.01)
    active -= 1
    return name.upper()

files = ["a", "b", "c", "d", "e"]

# your code here


### Q13 · Real-life

A kitchen has one order taker and one cook. The order taker puts every
dish of `orders` on an `asyncio.Queue`, then a `None`. The cook takes dishes
until `None` and prints `Cooked` plus the dish. Run both at the same time.

**Expected output:**

```text
Cooked pizza
Cooked salad
Cooked soup
```

<details><summary><b>Solution</b></summary>

```python
async def order_taker(queue):
    for dish in orders:
        await queue.put(dish)
    await queue.put(None)

async def cook(queue):
    while True:
        dish = await queue.get()
        if dish is None:
            break
        print("Cooked", dish)

queue = asyncio.Queue()
await asyncio.gather(order_taker(queue), cook(queue))
```

</details>

In [ ]:
import asyncio

orders = ["pizza", "salad", "soup"]

# your code here


### Q14 · Build

Write an **async context manager** class `Connection`. Entering it waits
0.001 seconds and records `open` in `events`; leaving records `close`. It
has an `async` method `query(text)` that records `query: ` plus the text.
Use it in an `async with` block that runs two queries, then print `events`.

**Expected output:**

```text
['open', 'query: SELECT 1', 'query: SELECT 2', 'close']
```

<details><summary><b>Solution</b></summary>

```python
class Connection:
    async def __aenter__(self):
        await asyncio.sleep(0.001)
        events.append("open")
        return self

    async def __aexit__(self, exc_type, exc, traceback):
        events.append("close")

    async def query(self, text):
        events.append("query: " + text)

async with Connection() as connection:
    await connection.query("SELECT 1")
    await connection.query("SELECT 2")
print(events)
```

</details>

In [ ]:
import asyncio

events = []

# your code here


### Q15 · Real-life

`flaky_api` fails with a `ConnectionError` the first two times it is called.
Write a coroutine `retry(func, attempts)` that awaits `func()` up to
`attempts` times and returns the first success; if the last attempt also
fails, the error must propagate. Call it with 5 attempts and print the
result, then the number of calls made.

**Expected output:**

```text
payload
Calls: 3
```

<details><summary><b>Solution</b></summary>

```python
async def retry(func, attempts):
    for attempt in range(1, attempts + 1):
        try:
            return await func()
        except ConnectionError:
            if attempt == attempts:
                raise

print(await retry(flaky_api, 5))
print("Calls:", calls)
```

</details>

In [ ]:
import asyncio

calls = 0

async def flaky_api():
    global calls
    calls += 1
    await asyncio.sleep(0.001)
    if calls < 3:
        raise ConnectionError("try again")
    return "payload"

# your code here


---

## Puzzles & builds · 8 to 15 lines

### Q16 · Puzzle

`add_one` reads the counter, lets other threads run, then writes back, so
parallel calls lose updates. Without changing `add_one`, run 5 threads that
each call it 20 times, and make the final `counter` exactly 100 by
protecting the calls with a lock. Print the counter.

**Expected output:**

```text
100
```

<details><summary><b>Solution</b></summary>

```python
lock = threading.Lock()

def add_many():
    for _ in range(20):
        with lock:
            add_one()

threads = [threading.Thread(target=add_many) for _ in range(5)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(counter)
```

</details>

In [ ]:
import threading
import time

counter = 0

def add_one():
    global counter
    current = counter
    time.sleep(0)
    counter = current + 1

# your code here


### Q17 · Puzzle

Two players take strict turns: `ping` goes first, then `pong`, three rounds
each. Use two `asyncio.Event` objects so each player waits for its turn
(and clears the event when its turn starts). Each turn appends the player's
name to `log`. Run both players together and print `log`.

**Expected output:**

```text
['ping', 'pong', 'ping', 'pong', 'ping', 'pong']
```

<details><summary><b>Solution</b></summary>

```python
async def ping():
    for _ in range(3):
        await ping_turn.wait()
        ping_turn.clear()
        log.append("ping")
        pong_turn.set()

async def pong():
    for _ in range(3):
        await pong_turn.wait()
        pong_turn.clear()
        log.append("pong")
        ping_turn.set()

ping_turn.set()
await asyncio.gather(ping(), pong())
print(log)
```

</details>

In [ ]:
import asyncio

log = []
ping_turn = asyncio.Event()
pong_turn = asyncio.Event()

# your code here


### Q18 · Real-life

Build a polite page fetcher. `fetch_with_retry(url)` may only run while
holding a `Semaphore(2)`, gives each attempt at most 0.02 seconds
(`asyncio.timeout`), and tries at most twice. It returns the page, or
`gave up` if both attempts time out. Fetch every url concurrently and
print one line per url, in order: the url, an arrow `->`, the result.

**Expected output:**

```text
a.com -> <html of a.com>
flaky.com -> <html of flaky.com>
slow.com -> gave up
b.com -> <html of b.com>
```

<details><summary><b>Solution</b></summary>

```python
limit = asyncio.Semaphore(2)

async def fetch_with_retry(url):
    async with limit:
        for attempt in (1, 2):
            try:
                async with asyncio.timeout(0.02):
                    return await fake_fetch(url, attempt)
            except TimeoutError:
                pass
        return "gave up"

pages = await asyncio.gather(*(fetch_with_retry(url) for url in urls))
for url, page in zip(urls, pages):
    print(url, "->", page)
```

</details>

In [ ]:
import asyncio

async def fake_fetch(url, attempt):
    if url == "slow.com" or (url == "flaky.com" and attempt == 1):
        await asyncio.sleep(0.05)
    else:
        await asyncio.sleep(0.001)
    return "<html of " + url + ">"

urls = ["a.com", "flaky.com", "slow.com", "b.com"]

# your code here


### Q19 · Real-life

Seven tickets are left and ten customers (threads) all try to buy one at
the same moment. Write a class `Stock` with a `count` and a method
`reserve()` that returns `True` and takes one ticket if any are left,
otherwise `False`. Between checking and taking, call `time.sleep(0)` to
simulate work, and make `reserve` safe with a lock. Start the ten customers,
wait for them, and print how many reservations succeeded and how many
tickets are left.

**Expected output:**

```text
Reserved: 7
Left: 0
```

<details><summary><b>Solution</b></summary>

```python
class Stock:
    def __init__(self, count):
        self.count = count
        self.lock = threading.Lock()

    def reserve(self):
        with self.lock:
            if self.count > 0:
                time.sleep(0)
                self.count -= 1
                return True
            return False

stock = Stock(7)

def customer():
    results.append(stock.reserve())

threads = [threading.Thread(target=customer) for _ in range(10)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print("Reserved:", results.count(True))
print("Left:", stock.count)
```

</details>

In [ ]:
import threading
import time

results = []

# your code here


---

## Q20 · Story project: Async Site Crawler

You're testing a link checker for a small website. There is no network:
`SITE` maps each page to the pages it links to, and `fetch` pretends to
download a page. Real crawlers are mostly waiting, which is exactly what
`asyncio` is good at. Every step builds on the ones before.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
SITE = {
    "/": ["/about", "/blog", "/shop"],
    "/about": ["/", "/team"],
    "/team": [],
    "/blog": ["/blog/1", "/missing"],
    "/blog/1": ["/blog/2"],
    "/blog/2": ["/"],
    "/shop": ["/shop/cart", "/slow"],
    "/shop/cart": [],
    "/slow": ["/hidden"],
}
DELAYS = {"/slow": 0.05}      # every other page answers in 0.001 s


### Step 1

Write `async def fetch(url)`: wait `DELAYS.get(url, 0.001)` seconds (`asyncio.sleep`), raise `FileNotFoundError(url)` if the page is not in `SITE`, otherwise return its list of links. Print the links of `/about`.

**Expected output:**

```text
['/', '/team']
```

<details><summary><b>Solution</b></summary>

```python
import asyncio
async def fetch(url):
    await asyncio.sleep(DELAYS.get(url, 0.001))
    if url not in SITE:
        raise FileNotFoundError(url)
    return SITE[url]
print(await fetch("/about"))
```

</details>

In [ ]:
# your code here


### Step 2

Write `async def fetch_many(urls)` that fetches all the urls concurrently without letting one failure cancel the rest, and returns a dict from url to result (the links, or the exception). Call it with `/`, `/missing` and `/team`, and print one line per url: the url and either the number of links or `ERROR`.

**Expected output:**

```text
/ 3
/missing ERROR
/team 0
```

<details><summary><b>Solution</b></summary>

```python
async def fetch_many(urls):
    results = await asyncio.gather(*(fetch(url) for url in urls), return_exceptions=True)
    return dict(zip(urls, results))
pages = await fetch_many(["/", "/missing", "/team"])
for url, result in pages.items():
    print(url, "ERROR" if isinstance(result, Exception) else len(result))
```

</details>

In [ ]:
# your code here


### Step 3

Write `async def crawl(start, limit)` that visits the site level by level (all pages of one level concurrently, at most `limit` at a time using a `Semaphore`) and keeps a list `seen` of every url found, in discovery order, and a list `broken` of urls that could not be fetched. Also record in a dict `stats` the highest number of fetches active at once under the key `peak`. Crawl from `/` with limit 2 and print the number of pages found, the sorted broken urls, and the peak, labelled like the expected output.

**Expected output:**

```text
Pages found: 11
Broken: ['/hidden', '/missing']
Peak: 2
```

<details><summary><b>Solution</b></summary>

```python
stats = {"active": 0, "peak": 0}
async def limited_fetch(url, limit):
    async with limit:
        stats["active"] += 1
        stats["peak"] = max(stats["peak"], stats["active"])
        try:
            return await fetch(url)
        finally:
            stats["active"] -= 1
async def crawl(start, limit):
    gate = asyncio.Semaphore(limit)
    seen = [start]
    broken = []
    level = [start]
    while level:
        results = await asyncio.gather(*(limited_fetch(url, gate) for url in level), return_exceptions=True)
        next_level = []
        for url, result in zip(level, results):
            if isinstance(result, Exception):
                broken.append(url)
                continue
            for link in result:
                if link not in seen:
                    seen.append(link)
                    next_level.append(link)
        level = next_level
    return seen, broken
seen, broken = await crawl("/", 2)
print("Pages found:", len(seen))
print("Broken:", sorted(broken))
print("Peak:", stats["peak"])
```

</details>

In [ ]:
# your code here


### Step 4

A real checker must not wait forever. Write `async def check(url, gate)` that, while holding the semaphore `gate`, fetches the url with a 0.02 second `asyncio.timeout` and returns `ok`, `broken` (page not found) or `timeout`. Check every url in `seen` with a `Semaphore(3)`, then print every url whose status is not `ok` as `url status` (sorted by url), and finally how many were ok.

**Expected output:**

```text
/hidden broken
/missing broken
/slow timeout
OK pages: 8
```

<details><summary><b>Solution</b></summary>

```python
async def check(url, gate):
    async with gate:
        try:
            async with asyncio.timeout(0.02):
                await fetch(url)
            return "ok"
        except FileNotFoundError:
            return "broken"
        except TimeoutError:
            return "timeout"
gate = asyncio.Semaphore(3)
statuses = await asyncio.gather(*(check(url, gate) for url in seen))
for url, status in sorted(zip(seen, statuses)):
    if status != "ok":
        print(url, status)
print("OK pages:", statuses.count("ok"))
```

</details>

In [ ]:
# your code here


### Step 5

Finish with the report. Write an async generator `healthy(urls, statuses)` that yields the urls whose status is `ok` (await `asyncio.sleep(0)` before each yield). Collect them with an `async for` into a list, then use a `TaskGroup` to fetch all those pages at once. Print each healthy page (sorted) with its number of links, then the page with the most links.

**Expected output:**

```text
/ 3
/about 2
/blog 2
/blog/1 1
/blog/2 1
/shop 2
/shop/cart 0
/team 0
Most links: /
```

<details><summary><b>Solution</b></summary>

```python
async def healthy(urls, statuses):
    for url, status in zip(urls, statuses):
        if status == "ok":
            await asyncio.sleep(0)
            yield url
good = []
async for url in healthy(seen, statuses):
    good.append(url)
async with asyncio.TaskGroup() as group:
    tasks = {url: group.create_task(fetch(url)) for url in good}
counts = {url: len(task.result()) for url, task in tasks.items()}
for url in sorted(counts):
    print(url, counts[url])
print("Most links:", max(counts, key=counts.get))
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB28 · Testing Debugging Logging** in `advanced/`.